In [2]:
def V04(L1,L2,L3,L4):
    return 0.5*(4*math.pi**2 + L1**2 + L2**2 + L3**2 + L4**2)

In [3]:
def masses_from_lengths(p):
    Λu, Λd, Λ0, theta = p
    # up-sector lengths
    Lu  = Λu*math.sqrt(mu_ref/mt_ref)
    Lc  = Λu*math.sqrt(mc_ref/mt_ref)
    Lt  = Λu                            # by definition
    # down-sector lengths
    Ld  = Λd*math.sqrt(md_ref/mb_ref)
    Ls  = Λd*math.sqrt(ms_ref/mb_ref)
    Lb  = Λd
    # extra glued boundary (common length Λ0)
    Vup   = V04(Lu,Lc,Lt,Λ0)
    Vdown = V04(Ld,Ls,Lb,Λ0)
    # identify mass ∝ boundary-squared
    mup = np.array([Lu**2,Lc**2,Lt**2])
    mdp = np.array([Ld**2,Ls**2,Lb**2])
    return np.concatenate([mup,mdp]), theta

In [4]:
# ---- Minimal hybrid (poly + exp) quark-mass fit -----------------------
import math, numpy as np
from scipy.optimize import minimize

# ---------- PDG reference masses (MSbar or pole mixed; illustrative) ---
m_ref = dict(u=0.00216, d=0.00467, s=0.093, c=1.2735, b=4.188, t=172.76)  # GeV
order  = ['u','c','t','d','s','b']          # up-sector first, then down

# ---------- One-loop alpha_s (nf=5) ------------------------------------
def alpha_s(mu, Lambda=0.22):
    b0 = (33 - 2*5)/(12*math.pi)
    return 1.0/(b0*math.log(mu**2/Lambda**2))

# ---------- Geometric term f(L)  ---------------------------------------
def geom_term(q, L, c0, c1, c2):
    """Quadratic for light; quad+exp for c,b,t."""
    poly = c0 + c1*L + c2*L**2
    if q in {'c','b','t'}:
        A = c1**2/(2*c2)          # from your k,A relations
        k = 2*c2/c1
        return poly + A*math.exp(k*L)
    return poly

# ---------- Model masses ------------------------------------------------
def model_masses(params):
    Λu, Λd, c1, c2 = params
    c0 = 0.0                      # keep constant term zero for now
    # length map: L = Λ * sqrt(m_ref / m_heavy_ref)
    heavy_u, heavy_d = m_ref['t'], m_ref['b']
    L = dict(
        u = Λu*math.sqrt(m_ref['u']/heavy_u),
        c = Λu*math.sqrt(m_ref['c']/heavy_u),
        t = Λu,
        d = Λd*math.sqrt(m_ref['d']/heavy_d),
        s = Λd*math.sqrt(m_ref['s']/heavy_d),
        b = Λd,
    )
    m_pred = {}
    for q in m_ref:
        fL   = geom_term(q, L[q], c0, c1, c2)
        # one-loop running factor at same scale cancels → omit
        m_pred[q] = fL
    return np.array([m_pred[q] for q in order])

# ---------- Chi-square in log space ------------------------------------
log_ref = np.log([m_ref[q] for q in order])
def chi2(params):
    log_model = np.log(model_masses(params))
    return np.sum((log_model - log_ref)**2)

# ---------- Optimise (bounds keep poly stable) -------------------------
p0     = [1.0, 0.3, 1.0, 0.5]     # Λu, Λd, c1, c2
bounds = [(0.05, 5.) , (0.05, 5.), (0.1, 10.), (0.01,  5.)]
res    = minimize(chi2, p0, method='L-BFGS-B', bounds=bounds)

# ---------- Print results ----------------------------------------------
print("Fit status:", res.message)
Λu, Λd, c1, c2 = res.x
print(f"Λu={Λu:.3f}, Λd={Λd:.3f},  c1={c1:.3f}, c2={c2:.3f}")
print("\nquark  model(GeV)  PDG(GeV)  error(%)")
m_fit = model_masses(res.x)
for q, m_model, m_true in zip(order, m_fit, [m_ref[q] for q in order]):
    err = 100*(m_model - m_true)/m_true
    print(f"{q:>2}   {m_model:9.4g}  {m_true:8.4g}  {err:+6.2f}")
print("\nχ² =", res.fun)

Fit status: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
Λu=1.287, Λd=0.461,  c1=0.874, c2=2.374

quark  model(GeV)  PDG(GeV)  error(%)
 u    0.004027   0.00216  +86.45
 c       0.419     1.274  -67.10
 t       179.1     172.8   +3.67
 d     0.01401   0.00467  +199.96
 s     0.07119     0.093  -23.46
 b       2.869     4.188  -31.51

χ² = 3.046711893910259


In [6]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import pandas as pd
from scipy.linalg import expm

class EnhancedPolynomialModel:
    """
    An enhanced implementation of the polynomial mass generation approach
    that includes all six quarks, CKM matrix calculations, and an improved top quark fit.
    """

    def __init__(self):
        """
        Initialize the enhanced polynomial model with lattice QCD parameters.
        """
        # Lattice QCD parameters (from provided papers and PDG)
        self.alpha_s_mz = 0.11803  # Strong coupling at Z-boson mass

        # Quark masses at reference scales (GeV)
        self.mc_mc = 1.2735  # Charm quark mass at its own scale
        self.mb_mb = 4.188  # Bottom quark mass at its own scale

        # PDG values for other quarks (GeV)
        self.mu_2gev = 0.00216  # Up quark mass at 2 GeV
        self.md_2gev = 0.00467  # Down quark mass at 2 GeV
        self.ms_2gev = 0.093  # Strange quark mass at 2 GeV
        self.mt_mt = 172.76  # Top quark mass at its own scale

        # Reference scales (GeV)
        self.mu_ref_light = 2.0  # Reference scale for light quarks (u, d, s)
        self.mu_c = self.mc_mc  # Reference scale for charm quark
        self.mu_b = self.mb_mb  # Reference scale for bottom quark
        self.mu_t = self.mt_mt  # Reference scale for top quark
        self.mz = 91.1876  # Z-boson mass

        # Initialize polynomial coefficients (to be optimized)
        self.poly_degree = 2  # Simplified to quadratic for stability

        # Separate coefficients for up-type and down-type quarks
        self.c_coeffs_up = np.ones(self.poly_degree + 1) * 0.1  # Initial guess
        self.c_coeffs_down = np.ones(self.poly_degree + 1) * 0.1  # Initial guess

        # Special top quark enhancement parameters (to be optimized)
        self.top_enhancement_factor = 10.0  # Initial guess
        self.top_exponent = 1.5  # Initial guess

        # Initialize geodesic lengths (to be optimized)
        self.L_u = 0.1  # Initial guess for up quark geodesic length
        self.L_d = 0.2  # Initial guess for down quark geodesic length
        self.L_s = 0.5  # Initial guess for strange quark geodesic length
        self.L_c = 1.0  # Initial guess for charm quark geodesic length
        self.L_b = 2.0  # Initial guess for bottom quark geodesic length
        self.L_t = 3.0  # Initial guess for top quark geodesic length

        # Initialize geodesic angles for CKM matrix (to be optimized)
        self.theta_12 = 0.2  # Initial guess for Cabibbo angle
        self.theta_13 = 0.01  # Initial guess for theta_13
        self.theta_23 = 0.04  # Initial guess for theta_23
        self.delta_cp = 1.2  # Initial guess for CP-violating phase

        # Experimental CKM matrix magnitudes (PDG 2022)
        self.ckm_exp = np.array([
            [0.97435, 0.22500, 0.00369],
            [0.22486, 0.97349, 0.04182],
            [0.00857, 0.04110, 0.99915]
        ])

        # Beta function coefficients for different Nf values
        self.beta0_nf3 = (11.0 - 2.0/3.0 * 3.0) / 4.0  # = 2.41667
        self.beta1_nf3 = (102.0 - 38.0/3.0 * 3.0) / 16.0  # ≈ 4.19

        self.beta0_nf4 = (11.0 - 2.0/3.0 * 4.0) / 4.0  # = 2.25
        self.beta1_nf4 = (102.0 - 38.0/3.0 * 4.0) / 16.0  # ≈ 3.79

        self.beta0_nf5 = (11.0 - 2.0/3.0 * 5.0) / 4.0  # = 2.0833
        self.beta1_nf5 = (102.0 - 38.0/3.0 * 5.0) / 16.0  # ≈ 3.4

        self.beta0_nf6 = (11.0 - 2.0/3.0 * 6.0) / 4.0  # = 1.9167
        self.beta1_nf6 = (102.0 - 38.0/3.0 * 6.0) / 16.0  # ≈ 3.0

        # Anomalous dimension coefficients
        self.gamma0 = 1.0
        self.gamma1_nf3 = (202.0/3.0 - 20.0/9.0 * 3.0) / 16.0  # ≈ 3.76
        self.gamma1_nf4 = (202.0/3.0 - 20.0/9.0 * 4.0) / 16.0  # ≈ 3.65
        self.gamma1_nf5 = (202.0/3.0 - 20.0/9.0 * 5.0) / 16.0  # ≈ 3.54
        self.gamma1_nf6 = (202.0/3.0 - 20.0/9.0 * 6.0) / 16.0  # ≈ 3.43

        # Generation scaling factors
        self.gen_scale = [1.0, 1.0, 1.0]  # To be optimized

        # Optimization status
        self.is_optimized = False

        # Results storage
        self.results = {}

        # Quark information
        self.quark_info = {
            'u': {'type': 'up', 'generation': 1, 'ref_mass': self.mu_2gev, 'ref_scale': self.mu_ref_light},
            'd': {'type': 'down', 'generation': 1, 'ref_mass': self.md_2gev, 'ref_scale': self.mu_ref_light},
            's': {'type': 'down', 'generation': 2, 'ref_mass': self.ms_2gev, 'ref_scale': self.mu_ref_light},
            'c': {'type': 'up', 'generation': 2, 'ref_mass': self.mc_mc, 'ref_scale': self.mu_c},
            'b': {'type': 'down', 'generation': 3, 'ref_mass': self.mb_mb, 'ref_scale': self.mu_b},
            't': {'type': 'up', 'generation': 3, 'ref_mass': self.mt_mt, 'ref_scale': self.mu_t}
        }

    def alpha_s(self, mu):
        """
        Calculate the strong coupling constant at scale mu using 2-loop approximation.

        Parameters:
        -----------
        mu : float
            Energy scale in GeV

        Returns:
        --------
        float
            Strong coupling constant at scale mu
        """
        # Determine number of active flavors
        if mu < 1.3:  # Below charm threshold
            nf = 3
            beta0 = self.beta0_nf3
            beta1 = self.beta1_nf3
        elif mu < 4.2:  # Below bottom threshold
            nf = 4
            beta0 = self.beta0_nf4
            beta1 = self.beta1_nf4
        elif mu < 173.0:  # Below top threshold
            nf = 5
            beta0 = self.beta0_nf5
            beta1 = self.beta1_nf5
        else:  # Above top threshold
            nf = 6
            beta0 = self.beta0_nf6
            beta1 = self.beta1_nf6

        # For scales close to Z-boson mass, use the reference value directly
        if abs(mu - self.mz) < 0.1:
            return self.alpha_s_mz

        # For scales above 10 GeV, use 2-loop approximation with reference at Z-boson mass
        if mu > 10.0:
            t = np.log(mu**2 / self.mz**2)
            alpha_s_inv = 1.0/self.alpha_s_mz + beta0*t - beta1*np.log(abs(t) + 1.0)
            return 1.0 / max(alpha_s_inv, 1e-10)  # Prevent division by zero

        # For lower scales, use a more stable approximation
        # Based on PDG parameterization
        if mu < 10.0:
            if mu < 1.0:
                # Freeze coupling at 1 GeV to avoid unphysical behavior
                mu_eff = 1.0
            else:
                mu_eff = mu

            # Approximate values based on PDG
            if nf == 3:
                return 0.35 - 0.05 * np.log(mu_eff)  # Approximate for low scales
            elif nf == 4:
                return 0.25 - 0.02 * np.log(mu_eff)  # Approximate for medium scales
            else:  # nf == 5
                return 0.20 - 0.01 * np.log(mu_eff)  # Approximate for higher scales

        # Fallback (should not reach here)
        return self.alpha_s_mz

    def running_mass(self, m_ref, mu_ref, mu, nf):
        """
        Calculate the running mass at scale mu.

        Parameters:
        -----------
        m_ref : float
            Reference mass in GeV
        mu_ref : float
            Reference scale in GeV
        mu : float
            Target scale in GeV
        nf : int
            Number of active flavors

        Returns:
        --------
        float
            Running mass at scale mu
        """
        # Select appropriate anomalous dimension coefficients
        if nf == 3:
            gamma0 = self.gamma0
            gamma1 = self.gamma1_nf3
            beta0 = self.beta0_nf3
        elif nf == 4:
            gamma0 = self.gamma0
            gamma1 = self.gamma1_nf4
            beta0 = self.beta0_nf4
        elif nf == 5:
            gamma0 = self.gamma0
            gamma1 = self.gamma1_nf5
            beta0 = self.beta0_nf5
        else:  # nf == 6
            gamma0 = self.gamma0
            gamma1 = self.gamma1_nf6
            beta0 = self.beta0_nf6

        # For scales very close to reference scale, return reference mass
        if abs(mu - mu_ref) < 0.01:
            return m_ref

        # Calculate alpha_s at both scales
        alpha_ref = self.alpha_s(mu_ref)
        alpha_mu = self.alpha_s(mu)

        # Ensure positive values
        alpha_ref = max(alpha_ref, 0.01)
        alpha_mu = max(alpha_mu, 0.01)

        # Use a simplified power-law approximation for stability
        # m(mu) = m(mu_ref) * (alpha_s(mu)/alpha_s(mu_ref))^(gamma0/(2*beta0))
        power = gamma0 / (2 * beta0)
        ratio = alpha_mu / alpha_ref

        # Calculate running mass with safeguards
        try:
            m_mu = m_ref * ratio**power

            # Apply bounds to prevent unphysical values
            if mu > mu_ref:  # Running to higher scale
                # Mass should decrease with increasing scale
                m_mu = min(m_mu, m_ref)
                # But not too much
                m_mu = max(m_mu, m_ref * 0.5)
            else:  # Running to lower scale
                # Mass should increase with decreasing scale
                m_mu = max(m_mu, m_ref)
                # But not too much
                m_mu = min(m_mu, m_ref * 2.0)

            return m_mu

        except (ValueError, ZeroDivisionError, OverflowError):
            # Fallback to linear approximation in log(mu)
            if mu > mu_ref:
                # Decrease by ~10% per decade of energy
                return m_ref * (1.0 - 0.1 * np.log10(mu / mu_ref))
            else:
                # Increase by ~10% per decade of energy
                return m_ref * (1.0 + 0.1 * np.log10(mu_ref / mu))

    def calculate_mass(self, quark, L=None):
        """
        Calculate quark mass from geodesic length using polynomial approach.

        Parameters:
        -----------
        quark : str
            Quark name ('u', 'd', 's', 'c', 'b', 't')
        L : float, optional
            Geodesic length. If None, uses the model's optimized length for the quark.

        Returns:
        --------
        float
            Quark mass in GeV at reference scale
        """
        # Get quark information
        quark_type = self.quark_info[quark]['type']
        generation = self.quark_info[quark]['generation']

        # Get appropriate coefficients
        if quark_type == 'up':
            coeffs = self.c_coeffs_up
        else:  # quark_type == 'down'
            coeffs = self.c_coeffs_down

        # Get geodesic length
        if L is None:
            L = getattr(self, f'L_{quark}')

        # Calculate base mass using polynomial
        mass = 0.0
        for i in range(self.poly_degree + 1):
            mass += coeffs[i] * L**i

        # Apply generation-specific scaling
        mass *= self.gen_scale[generation - 1]

        # Special enhancement for top quark
        if quark == 't':
            # Add exponential enhancement term for top quark
            enhancement = self.top_enhancement_factor * np.exp(self.top_exponent * L)
            mass += enhancement

        return mass

    def calculate_ckm_matrix(self):
        """
        Calculate the CKM matrix using the standard parameterization.

        Returns:
        --------
        numpy.ndarray
            3x3 CKM matrix
        """
        # Calculate sines and cosines
        s12 = np.sin(self.theta_12)
        c12 = np.cos(self.theta_12)
        s13 = np.sin(self.theta_13)
        c13 = np.cos(self.theta_13)
        s23 = np.sin(self.theta_23)
        c23 = np.cos(self.theta_23)

        # Complex phase
        delta = self.delta_cp

        # Construct CKM matrix using standard parameterization
        ckm = np.array([
            [c12 * c13,
             s12 * c13,
             s13 * np.exp(-1j * delta)],

            [-s12 * c23 - c12 * s23 * s13 * np.exp(1j * delta),
             c12 * c23 - s12 * s23 * s13 * np.exp(1j * delta),
             s23 * c13],

            [s12 * s23 - c12 * c23 * s13 * np.exp(1j * delta),
             -c12 * s23 - s12 * c23 * s13 * np.exp(1j * delta),
             c23 * c13]
        ])

        return ckm

    def calculate_ckm_from_geodesics(self):
        """
        Calculate the CKM matrix from geodesic lengths using a geometric approach.

        Returns:
        --------
        numpy.ndarray
            3x3 CKM matrix
        """
        # Get geodesic lengths
        L_u = self.L_u
        L_d = self.L_d
        L_s = self.L_s
        L_c = self.L_c
        L_b = self.L_b
        L_t = self.L_t

        # Calculate geodesic distances between up-type and down-type quarks
        # Using a simplified distance formula based on geodesic lengths
        distances = np.zeros((3, 3))

        # Up-type quarks (u, c, t)
        up_quarks = [L_u, L_c, L_t]
        # Down-type quarks (d, s, b)
        down_quarks = [L_d, L_s, L_b]

        # Calculate distances between each pair
        for i, L_up in enumerate(up_quarks):
            for j, L_down in enumerate(down_quarks):
                # Simple distance formula based on geodesic lengths
                distances[i, j] = abs(L_up - L_down) / (1 + L_up * L_down)

        # Convert distances to mixing angles
        # Smaller distance = stronger mixing
        max_distance = np.max(distances)
        normalized_distances = distances / max_distance

        # Convert to mixing strengths (inverse of normalized distances)
        mixing_strengths = 1.0 / (1.0 + normalized_distances)

        # Normalize to ensure unitarity
        for i in range(3):
            row_sum = np.sum(mixing_strengths[i, :])
            mixing_strengths[i, :] /= row_sum

        # Ensure columns are also normalized
        for j in range(3):
            col_sum = np.sum(mixing_strengths[:, j])
            mixing_strengths[:, j] /= col_sum

        # Final normalization to ensure unitarity
        total_sum = np.sum(mixing_strengths)
        mixing_strengths = 3 * mixing_strengths / total_sum

        return mixing_strengths

    def optimize_parameters(self):
        """
        Optimize model parameters to match lattice QCD, PDG values, and CKM matrix.

        Returns:
        --------
        dict
            Optimization results
        """
        # Define objective function
        def objective(params):
            # Extract parameters
            self.c_coeffs_up = params[0:3]
            self.c_coeffs_down = params[3:6]
            self.top_enhancement_factor = params[6]
            self.top_exponent = params[7]
            self.L_u = params[8]
            self.L_d = params[9]
            self.L_s = params[10]
            self.L_c = params[11]
            self.L_b = params[12]
            self.L_t = params[13]
            self.gen_scale = params[14:17]
            self.theta_12 = params[17]
            self.theta_13 = params[18]
            self.theta_23 = params[19]
            self.delta_cp = params[20]

            # Calculate masses at reference scales
            masses = {}
            errors = {}

            for quark in self.quark_info:
                ref_mass = self.quark_info[quark]['ref_mass']
                masses[quark] = self.calculate_mass(quark)
                errors[quark] = ((masses[quark] - ref_mass) / ref_mass)**2

            # Calculate CKM matrix
            ckm = self.calculate_ckm_matrix()
            ckm_mag = np.abs(ckm)

            # Calculate CKM matrix errors
            ckm_errors = np.sum((ckm_mag - self.ckm_exp)**2)

            # Total error with weighting
            # Give more weight to top quark and CKM matrix
            total_error = (
                errors['u'] + errors['d'] + errors['s'] +
                5 * errors['c'] + 5 * errors['b'] + 20 * errors['t'] +
                10 * ckm_errors
            )

            # Add regularization to prevent extreme values
            regularization = 0.01 * (
                np.sum(self.c_coeffs_up**2) +
                np.sum(self.c_coeffs_down**2) +
                self.top_enhancement_factor**2 +
                self.top_exponent**2 +
                np.sum((np.array([self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t]) -
                       np.array([0.1, 0.2, 0.5, 1.0, 2.0, 3.0]))**2)
            )

            return total_error + regularization

        # Initial guess
        initial_guess = np.concatenate([
            self.c_coeffs_up,
            self.c_coeffs_down,
            [self.top_enhancement_factor, self.top_exponent],
            [self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t],
            self.gen_scale,
            [self.theta_12, self.theta_13, self.theta_23, self.delta_cp]
        ])

        # Bounds for parameters
        bounds = []
        # Coefficients for up-type quarks
        bounds.extend([(0.001, 10.0) for _ in range(self.poly_degree + 1)])
        # Coefficients for down-type quarks
        bounds.extend([(0.001, 10.0) for _ in range(self.poly_degree + 1)])
        # Top quark enhancement parameters
        bounds.append((1.0, 1000.0))  # top_enhancement_factor
        bounds.append((0.1, 5.0))     # top_exponent
        # Geodesic lengths
        bounds.append((0.01, 0.5))  # L_u
        bounds.append((0.01, 0.5))  # L_d
        bounds.append((0.1, 1.0))   # L_s
        bounds.append((0.5, 2.0))   # L_c
        bounds.append((1.0, 3.0))   # L_b
        bounds.append((2.0, 5.0))   # L_t
        # Generation scaling factors
        bounds.append((0.001, 10.0))  # gen_scale[0]
        bounds.append((0.001, 1.0))   # gen_scale[1]
        bounds.append((0.001, 0.1))   # gen_scale[2]
        # CKM parameters
        bounds.append((0.1, 0.3))    # theta_12 (Cabibbo angle)
        bounds.append((0.001, 0.05))  # theta_13
        bounds.append((0.01, 0.1))    # theta_23
        bounds.append((0.0, 2*np.pi)) # delta_cp

        # Perform optimization
        result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B')

        # Update parameters with optimized values
        self.c_coeffs_up = result.x[0:3]
        self.c_coeffs_down = result.x[3:6]
        self.top_enhancement_factor = result.x[6]
        self.top_exponent = result.x[7]
        self.L_u = result.x[8]
        self.L_d = result.x[9]
        self.L_s = result.x[10]
        self.L_c = result.x[11]
        self.L_b = result.x[12]
        self.L_t = result.x[13]
        self.gen_scale = result.x[14:17]
        self.theta_12 = result.x[17]
        self.theta_13 = result.x[18]
        self.theta_23 = result.x[19]
        self.delta_cp = result.x[20]

        # Calculate masses and errors
        masses = {}
        errors = {}

        for quark in self.quark_info:
            ref_mass = self.quark_info[quark]['ref_mass']
            masses[quark] = self.calculate_mass(quark)
            errors[quark] = abs((masses[quark] - ref_mass) / ref_mass) * 100  # Percentage

        # Calculate CKM matrix
        ckm = self.calculate_ckm_matrix()
        ckm_mag = np.abs(ckm)

        # Calculate CKM matrix errors
        ckm_errors = np.abs((ckm_mag - self.ckm_exp) / self.ckm_exp) * 100  # Percentage

        # Store results
        self.results = {
            'c_coeffs_up': self.c_coeffs_up,
            'c_coeffs_down': self.c_coeffs_down,
            'top_enhancement_factor': self.top_enhancement_factor,
            'top_exponent': self.top_exponent,
            'L_u': self.L_u,
            'L_d': self.L_d,
            'L_s': self.L_s,
            'L_c': self.L_c,
            'L_b': self.L_b,
            'L_t': self.L_t,
            'gen_scale': self.gen_scale,
            'theta_12': self.theta_12,
            'theta_13': self.theta_13,
            'theta_23': self.theta_23,
            'delta_cp': self.delta_cp,
            'masses': masses,
            'errors': errors,
            'ckm': ckm_mag,
            'ckm_exp': self.ckm_exp,
            'ckm_errors': ckm_errors,
            'success': result.success,
            'message': result.message
        }

        self.is_optimized = True

        return self.results

    def calculate_running_masses(self, mu_values):
        """
        Calculate running masses at different energy scales.

        Parameters:
        -----------
        mu_values : list
            List of energy scales in GeV

        Returns:
        --------
        dict
            Dictionary of running masses
        """
        if not self.is_optimized:
            self.optimize_parameters()

        running_masses = {'mu': mu_values}

        for quark in self.quark_info:
            running_masses[quark] = []
            ref_mass = self.results['masses'][quark]
            ref_scale = self.quark_info[quark]['ref_scale']

            for mu in mu_values:
                # Determine number of active flavors
                if mu < 1.3:  # Below charm threshold
                    nf = 3
                elif mu < 4.2:  # Below bottom threshold
                    nf = 4
                elif mu < 173.0:  # Below top threshold
                    nf = 5
                else:  # Above top threshold
                    nf = 6

                # Calculate running mass
                m_mu = self.running_mass(ref_mass, ref_scale, mu, nf)
                running_masses[quark].append(m_mu)

        return running_masses

    def calculate_alpha_s_values(self, mu_values):
        """
        Calculate strong coupling constant at different energy scales.

        Parameters:
        -----------
        mu_values : list
            List of energy scales in GeV

        Returns:
        --------
        dict
            Dictionary of alpha_s values
        """
        alpha_s_values = {'mu': mu_values, 'alpha_s': []}

        for mu in mu_values:
            alpha_s_values['alpha_s'].append(self.alpha_s(mu))

        return alpha_s_values

    def generate_report(self):
        """
        Generate a report of the model results.

        Returns:
        --------
        str
            Path to the report file
        """
        if not self.is_optimized:
            self.optimize_parameters()

        # Create report file
        report_path = "enhanced_model_with_ckm_report.md"
        with open(report_path, 'w') as f:
            f.write("# Enhanced Polynomial Model with CKM Matrix Report\n\n")

            f.write("## Optimized Parameters\n\n")
            f.write("### Polynomial Coefficients for Up-type Quarks\n\n")
            for i, c in enumerate(self.c_coeffs_up):
                f.write(f"c_{i} = {c:.6f} GeV\n")

            f.write("\n### Polynomial Coefficients for Down-type Quarks\n\n")
            for i, c in enumerate(self.c_coeffs_down):
                f.write(f"c_{i} = {c:.6f} GeV\n")

            f.write("\n### Top Quark Enhancement Parameters\n\n")
            f.write(f"Enhancement Factor = {self.top_enhancement_factor:.6f}\n")
            f.write(f"Exponent = {self.top_exponent:.6f}\n")

            f.write("\n### Generation Scaling Factors\n\n")
            for i, s in enumerate(self.gen_scale):
                f.write(f"Generation {i+1}: {s:.6f}\n")

            f.write("\n### Geodesic Lengths\n\n")
            f.write(f"L_u = {self.L_u:.6f}\n")
            f.write(f"L_d = {self.L_d:.6f}\n")
            f.write(f"L_s = {self.L_s:.6f}\n")
            f.write(f"L_c = {self.L_c:.6f}\n")
            f.write(f"L_b = {self.L_b:.6f}\n")
            f.write(f"L_t = {self.L_t:.6f}\n")

            f.write("\n### CKM Matrix Parameters\n\n")
            f.write(f"θ₁₂ = {self.theta_12:.6f} rad = {np.degrees(self.theta_12):.4f}°\n")
            f.write(f"θ₁₃ = {self.theta_13:.6f} rad = {np.degrees(self.theta_13):.4f}°\n")
            f.write(f"θ₂₃ = {self.theta_23:.6f} rad = {np.degrees(self.theta_23):.4f}°\n")
            f.write(f"δ_CP = {self.delta_cp:.6f} rad = {np.degrees(self.delta_cp):.4f}°\n")

            f.write("\n## Mass Predictions at Reference Scales\n\n")
            f.write("| Quark | Reference Scale (GeV) | Predicted Mass (GeV) | Reference Value (GeV) | Error (%) |\n")
            f.write("|-------|----------------------|----------------------|----------------------|----------|\n")

            for quark in self.quark_info:
                ref_scale = self.quark_info[quark]['ref_scale']
                ref_mass = self.quark_info[quark]['ref_mass']
                pred_mass = self.results['masses'][quark]
                error = self.results['errors'][quark]
                f.write(f"| {quark} | {ref_scale:.4f} | {pred_mass:.6f} | {ref_mass:.6f} | {error:.4f} |\n")

            f.write("\n## CKM Matrix\n\n")
            f.write("### Predicted CKM Matrix (Magnitudes)\n\n")
            f.write("```\n")
            for i in range(3):
                f.write("[ ")
                for j in range(3):
                    f.write(f"{self.results['ckm'][i, j]:.6f} ")
                f.write("]\n")
            f.write("```\n\n")

            f.write("### Experimental CKM Matrix (Magnitudes)\n\n")
            f.write("```\n")
            for i in range(3):
                f.write("[ ")
                for j in range(3):
                    f.write(f"{self.ckm_exp[i, j]:.6f} ")
                f.write("]\n")
            f.write("```\n\n")

            f.write("### CKM Matrix Errors (%)\n\n")
            f.write("```\n")
            for i in range(3):
                f.write("[ ")
                for j in range(3):
                    f.write(f"{self.results['ckm_errors'][i, j]:.4f} ")
                f.write("]\n")
            f.write("```\n\n")

            f.write("\n## Running Masses\n\n")
            mu_values = [1.0, 2.0, 5.0, 10.0, 91.1876, 173.0]
            running_masses = self.calculate_running_masses(mu_values)

            f.write("| μ (GeV) | m_u (GeV) | m_d (GeV) | m_s (GeV) | m_c (GeV) | m_b (GeV) | m_t (GeV) |\n")
            f.write("|---------|-----------|-----------|-----------|-----------|-----------|----------|\n")

            for i, mu in enumerate(running_masses['mu']):
                f.write(f"| {mu:.4f} | {running_masses['u'][i]:.6f} | {running_masses['d'][i]:.6f} | {running_masses['s'][i]:.6f} | {running_masses['c'][i]:.6f} | {running_masses['b'][i]:.6f} | {running_masses['t'][i]:.6f} |\n")

            f.write("\n## Strong Coupling Constant\n\n")
            alpha_s_values = self.calculate_alpha_s_values(mu_values)

            f.write("| μ (GeV) | α_s |\n")
            f.write("|---------|------|\n")

            for i, mu in enumerate(alpha_s_values['mu']):
                f.write(f"| {mu:.4f} | {alpha_s_values['alpha_s'][i]:.6f} |\n")

            f.write("\n## Optimization Status\n\n")
            f.write(f"Success: {self.results['success']}\n")
            f.write(f"Message: {self.results['message']}\n")

        return report_path

    def plot_running_masses(self):
        """
        Plot running masses as a function of energy scale.

        Returns:
        --------
        str
            Path to the plot file
        """
        if not self.is_optimized:
            self.optimize_parameters()

        # Generate data
        mu_values = np.logspace(0, 3, 100)  # 1 GeV to 1000 GeV
        running_masses = self.calculate_running_masses(mu_values)

        # Create plot
        plt.figure(figsize=(12, 8))

        # Plot light quarks
        plt.loglog(running_masses['mu'], running_masses['u'], 'r-', label='Up Quark')
        plt.loglog(running_masses['mu'], running_masses['d'], 'b-', label='Down Quark')
        plt.loglog(running_masses['mu'], running_masses['s'], 'g-', label='Strange Quark')

        # Plot heavy quarks
        plt.loglog(running_masses['mu'], running_masses['c'], 'c-', label='Charm Quark')
        plt.loglog(running_masses['mu'], running_masses['b'], 'm-', label='Bottom Quark')
        plt.loglog(running_masses['mu'], running_masses['t'], 'y-', label='Top Quark')

        # Add reference points
        plt.scatter([self.mu_ref_light], [self.mu_2gev], color='red', marker='o', s=50, label='_nolegend_')
        plt.scatter([self.mu_ref_light], [self.md_2gev], color='blue', marker='o', s=50, label='_nolegend_')
        plt.scatter([self.mu_ref_light], [self.ms_2gev], color='green', marker='o', s=50, label='_nolegend_')
        plt.scatter([self.mu_c], [self.mc_mc], color='cyan', marker='o', s=50, label='_nolegend_')
        plt.scatter([self.mu_b], [self.mb_mb], color='magenta', marker='o', s=50, label='_nolegend_')
        plt.scatter([self.mu_t], [self.mt_mt], color='yellow', marker='o', s=50, label='Reference Values')

        # Set labels and title
        plt.xlabel('Energy Scale μ (GeV)')
        plt.ylabel('Running Mass (GeV)')
        plt.title('Running Quark Masses')
        plt.grid(True, which='both', linestyle='--', alpha=0.7)
        plt.legend()

        # Save plot
        plot_path = "enhanced_running_masses_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        # Create a second plot with better visibility for light quarks
        plt.figure(figsize=(12, 8))

        # Plot light quarks only
        plt.loglog(running_masses['mu'], running_masses['u'], 'r-', label='Up Quark')
        plt.loglog(running_masses['mu'], running_masses['d'], 'b-', label='Down Quark')
        plt.loglog(running_masses['mu'], running_masses['s'], 'g-', label='Strange Quark')

        # Add reference points
        plt.scatter([self.mu_ref_light], [self.mu_2gev], color='red', marker='o', s=50, label='_nolegend_')
        plt.scatter([self.mu_ref_light], [self.md_2gev], color='blue', marker='o', s=50, label='_nolegend_')
        plt.scatter([self.mu_ref_light], [self.ms_2gev], color='green', marker='o', s=50, label='Reference Values')

        # Set labels and title
        plt.xlabel('Energy Scale μ (GeV)')
        plt.ylabel('Running Mass (GeV)')
        plt.title('Running Masses of Light Quarks')
        plt.grid(True, which='both', linestyle='--', alpha=0.7)
        plt.legend()

        # Save plot
        light_plot_path = "enhanced_light_quarks_plot.png"
        plt.savefig(light_plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path, light_plot_path

    def plot_alpha_s(self):
        """
        Plot strong coupling constant as a function of energy scale.

        Returns:
        --------
        str
            Path to the plot file
        """
        # Generate data
        mu_values = np.logspace(0, 3, 100)  # 1 GeV to 1000 GeV
        alpha_s_values = self.calculate_alpha_s_values(mu_values)

        # Create plot
        plt.figure(figsize=(10, 6))
        plt.loglog(alpha_s_values['mu'], alpha_s_values['alpha_s'])

        # Add reference point
        plt.scatter([self.mz], [self.alpha_s_mz], color='red', marker='o', s=100, label='Lattice QCD')

        # Add flavor thresholds
        plt.axvline(x=1.3, color='c', linestyle='--', alpha=0.5, label='Charm Threshold')
        plt.axvline(x=4.2, color='m', linestyle='--', alpha=0.5, label='Bottom Threshold')
        plt.axvline(x=173.0, color='y', linestyle='--', alpha=0.5, label='Top Threshold')

        # Set labels and title
        plt.xlabel('Energy Scale μ (GeV)')
        plt.ylabel('Strong Coupling Constant α_s')
        plt.title('Running of the Strong Coupling Constant')
        plt.grid(True, which='both', linestyle='--', alpha=0.7)
        plt.legend()

        # Save plot
        plot_path = "enhanced_alpha_s_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path

    def plot_polynomial_functions(self):
        """
        Plot the polynomial mass generation functions.

        Returns:
        --------
        str
            Path to the plot file
        """
        if not self.is_optimized:
            self.optimize_parameters()

        # Generate data
        L_values = np.linspace(0, 5, 100)

        # Calculate masses for up-type quarks (without top enhancement)
        up_masses = np.zeros_like(L_values)
        for i, L in enumerate(L_values):
            up_masses[i] = np.sum([self.c_coeffs_up[j] * L**j for j in range(self.poly_degree + 1)])

        # Calculate masses for down-type quarks
        down_masses = np.zeros_like(L_values)
        for i, L in enumerate(L_values):
            down_masses[i] = np.sum([self.c_coeffs_down[j] * L**j for j in range(self.poly_degree + 1)])

        # Calculate top quark enhancement
        top_enhancement = np.zeros_like(L_values)
        for i, L in enumerate(L_values):
            top_enhancement[i] = self.top_enhancement_factor * np.exp(self.top_exponent * L)

        # Create plot
        plt.figure(figsize=(10, 6))
        plt.plot(L_values, up_masses, 'r-', label='Up-type Quarks (Base)')
        plt.plot(L_values, down_masses, 'b-', label='Down-type Quarks')
        plt.plot(L_values, up_masses + top_enhancement, 'y-', label='Up-type with Top Enhancement')

        # Add quark points
        plt.scatter([self.L_u], [self.calculate_mass('u')], color='r', marker='o', s=50, label='Up')
        plt.scatter([self.L_c], [self.calculate_mass('c')], color='r', marker='s', s=50, label='Charm')
        plt.scatter([self.L_t], [self.calculate_mass('t')], color='y', marker='^', s=50, label='Top')

        plt.scatter([self.L_d], [self.calculate_mass('d')], color='b', marker='o', s=50, label='Down')
        plt.scatter([self.L_s], [self.calculate_mass('s')], color='b', marker='s', s=50, label='Strange')
        plt.scatter([self.L_b], [self.calculate_mass('b')], color='b', marker='^', s=50, label='Bottom')

        # Set labels and title
        plt.xlabel('Geodesic Length L')
        plt.ylabel('Mass (GeV)')
        plt.title('Polynomial Mass Generation Functions')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()

        # Save plot
        plot_path = "enhanced_polynomial_functions_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        # Create a second plot with log scale for better visibility
        plt.figure(figsize=(10, 6))
        plt.semilogy(L_values, up_masses, 'r-', label='Up-type Quarks (Base)')
        plt.semilogy(L_values, down_masses, 'b-', label='Down-type Quarks')
        plt.semilogy(L_values, up_masses + top_enhancement, 'y-', label='Up-type with Top Enhancement')

        # Add quark points
        plt.scatter([self.L_u], [self.calculate_mass('u')], color='r', marker='o', s=50, label='Up')
        plt.scatter([self.L_c], [self.calculate_mass('c')], color='r', marker='s', s=50, label='Charm')
        plt.scatter([self.L_t], [self.calculate_mass('t')], color='y', marker='^', s=50, label='Top')

        plt.scatter([self.L_d], [self.calculate_mass('d')], color='b', marker='o', s=50, label='Down')
        plt.scatter([self.L_s], [self.calculate_mass('s')], color='b', marker='s', s=50, label='Strange')
        plt.scatter([self.L_b], [self.calculate_mass('b')], color='b', marker='^', s=50, label='Bottom')

        # Set labels and title
        plt.xlabel('Geodesic Length L')
        plt.ylabel('Mass (GeV) - Log Scale')
        plt.title('Polynomial Mass Generation Functions (Log Scale)')
        plt.grid(True, which='both', linestyle='--', alpha=0.7)
        plt.legend()

        # Save plot
        log_plot_path = "enhanced_polynomial_functions_log_plot.png"
        plt.savefig(log_plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path, log_plot_path

    def plot_geodesic_lengths(self):
        """
        Plot the geodesic lengths for all quarks.

        Returns:
        --------
        str
            Path to the plot file
        """
        if not self.is_optimized:
            self.optimize_parameters()

        # Create plot
        plt.figure(figsize=(10, 6))

        # Plot geodesic lengths
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        lengths = [self.L_u, self.L_d, self.L_s, self.L_c, self.L_b, self.L_t]
        colors = ['r', 'b', 'g', 'c', 'm', 'y']

        # Create bar chart
        plt.bar(quarks, lengths, color=colors)

        # Set labels and title
        plt.xlabel('Quark')
        plt.ylabel('Geodesic Length')
        plt.title('Geodesic Lengths for All Quarks')
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)

        # Save plot
        plot_path = "enhanced_geodesic_lengths_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path

    def plot_mass_hierarchy(self):
        """
        Plot the mass hierarchy for all quarks.

        Returns:
        --------
        str
            Path to the plot file
        """
        if not self.is_optimized:
            self.optimize_parameters()

        # Create plot
        plt.figure(figsize=(10, 6))

        # Plot masses
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        masses = [self.results['masses'][q] for q in quarks]
        colors = ['r', 'b', 'g', 'c', 'm', 'y']

        # Create bar chart with log scale
        plt.bar(quarks, masses, color=colors)
        plt.yscale('log')

        # Set labels and title
        plt.xlabel('Quark')
        plt.ylabel('Mass (GeV) - Log Scale')
        plt.title('Quark Mass Hierarchy')
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)

        # Save plot
        plot_path = "enhanced_mass_hierarchy_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path

    def plot_ckm_matrix(self):
        """
        Plot the CKM matrix.

        Returns:
        --------
        str
            Path to the plot file
        """
        if not self.is_optimized:
            self.optimize_parameters()

        # Create figure with multiple subplots
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # Plot 1: Predicted CKM matrix
        im1 = axes[0].imshow(self.results['ckm'], cmap='viridis', vmin=0, vmax=1)
        axes[0].set_title('Predicted CKM Matrix')

        # Add text annotations
        for i in range(3):
            for j in range(3):
                axes[0].text(j, i, f"{self.results['ckm'][i, j]:.4f}",
                           ha="center", va="center", color="w" if self.results['ckm'][i, j] < 0.5 else "k")

        # Plot 2: Experimental CKM matrix
        im2 = axes[1].imshow(self.ckm_exp, cmap='viridis', vmin=0, vmax=1)
        axes[1].set_title('Experimental CKM Matrix')

        # Add text annotations
        for i in range(3):
            for j in range(3):
                axes[1].text(j, i, f"{self.ckm_exp[i, j]:.4f}",
                           ha="center", va="center", color="w" if self.ckm_exp[i, j] < 0.5 else "k")

        # Plot 3: Error percentage
        im3 = axes[2].imshow(self.results['ckm_errors'], cmap='hot', vmin=0, vmax=20)
        axes[2].set_title('Error Percentage')

        # Add text annotations
        for i in range(3):
            for j in range(3):
                axes[2].text(j, i, f"{self.results['ckm_errors'][i, j]:.2f}%",
                           ha="center", va="center", color="w" if self.results['ckm_errors'][i, j] > 10 else "k")

        # Add labels to all subplots
        for ax in axes:
            ax.set_xticks([0, 1, 2])
            ax.set_yticks([0, 1, 2])
            ax.set_xticklabels(['d', 's', 'b'])
            ax.set_yticklabels(['u', 'c', 't'])
            ax.set_xlabel('Down-type Quarks')
            ax.set_ylabel('Up-type Quarks')

        # Add colorbars
        fig.colorbar(im1, ax=axes[0], label='Magnitude')
        fig.colorbar(im2, ax=axes[1], label='Magnitude')
        fig.colorbar(im3, ax=axes[2], label='Error (%)')

        plt.tight_layout()

        # Save plot
        plot_path = "enhanced_ckm_matrix_plot.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path

    def create_comprehensive_visualization(self):
        """
        Create a comprehensive visualization of the model results.

        Returns:
        --------
        str
            Path to the plot file
        """
        if not self.is_optimized:
            self.optimize_parameters()

        # Generate data
        mu_values = np.logspace(0, 3, 100)  # 1 GeV to 1000 GeV
        running_masses = self.calculate_running_masses(mu_values)
        alpha_s_values = self.calculate_alpha_s_values(mu_values)

        # Create figure with multiple subplots
        fig = plt.figure(figsize=(15, 12))

        # Plot 1: Running masses (all quarks)
        ax1 = fig.add_subplot(2, 2, 1)
        for quark in self.quark_info:
            ax1.loglog(running_masses['mu'], running_masses[quark], label=f'{quark} Quark')

        # Add reference points
        for quark in self.quark_info:
            ref_scale = self.quark_info[quark]['ref_scale']
            ref_mass = self.quark_info[quark]['ref_mass']
            ax1.scatter([ref_scale], [ref_mass], marker='o', s=30)

        ax1.set_xlabel('Energy Scale μ (GeV)')
        ax1.set_ylabel('Running Mass (GeV)')
        ax1.set_title('Running Quark Masses (All Quarks)')
        ax1.grid(True, which='both', linestyle='--', alpha=0.7)
        ax1.legend()

        # Plot 2: CKM matrix
        ax2 = fig.add_subplot(2, 2, 2)
        im = ax2.imshow(self.results['ckm'], cmap='viridis', vmin=0, vmax=1)
        ax2.set_title('CKM Matrix')

        # Add text annotations
        for i in range(3):
            for j in range(3):
                ax2.text(j, i, f"{self.results['ckm'][i, j]:.4f}",
                       ha="center", va="center", color="w" if self.results['ckm'][i, j] < 0.5 else "k")

        ax2.set_xticks([0, 1, 2])
        ax2.set_yticks([0, 1, 2])
        ax2.set_xticklabels(['d', 's', 'b'])
        ax2.set_yticklabels(['u', 'c', 't'])
        ax2.set_xlabel('Down-type Quarks')
        ax2.set_ylabel('Up-type Quarks')
        fig.colorbar(im, ax=ax2, label='Magnitude')

        # Plot 3: Polynomial functions
        ax3 = fig.add_subplot(2, 2, 3)

        # Generate data for polynomial functions
        L_values = np.linspace(0, 5, 100)

        # Calculate masses for up-type quarks (without top enhancement)
        up_masses = np.zeros_like(L_values)
        for i, L in enumerate(L_values):
            up_masses[i] = np.sum([self.c_coeffs_up[j] * L**j for j in range(self.poly_degree + 1)])

        # Calculate masses for down-type quarks
        down_masses = np.zeros_like(L_values)
        for i, L in enumerate(L_values):
            down_masses[i] = np.sum([self.c_coeffs_down[j] * L**j for j in range(self.poly_degree + 1)])

        # Calculate top quark enhancement
        top_enhancement = np.zeros_like(L_values)
        for i, L in enumerate(L_values):
            top_enhancement[i] = self.top_enhancement_factor * np.exp(self.top_exponent * L)

        ax3.semilogy(L_values, up_masses, 'r-', label='Up-type (Base)')
        ax3.semilogy(L_values, down_masses, 'b-', label='Down-type')
        ax3.semilogy(L_values, up_masses + top_enhancement, 'y-', label='With Top Enhancement')

        # Add quark points
        ax3.scatter([self.L_u], [self.calculate_mass('u')], color='r', marker='o', s=50, label='Up')
        ax3.scatter([self.L_c], [self.calculate_mass('c')], color='r', marker='s', s=50, label='Charm')
        ax3.scatter([self.L_t], [self.calculate_mass('t')], color='y', marker='^', s=50, label='Top')

        ax3.scatter([self.L_d], [self.calculate_mass('d')], color='b', marker='o', s=50, label='Down')
        ax3.scatter([self.L_s], [self.calculate_mass('s')], color='b', marker='s', s=50, label='Strange')
        ax3.scatter([self.L_b], [self.calculate_mass('b')], color='b', marker='^', s=50, label='Bottom')

        ax3.set_xlabel('Geodesic Length')
        ax3.set_ylabel('Mass (GeV)')
        ax3.set_title('Polynomial Mass Generation Functions')
        ax3.grid(True, which='both', linestyle='--', alpha=0.7)
        ax3.legend()

        # Plot 4: Mass hierarchy
        ax4 = fig.add_subplot(2, 2, 4)

        # Plot masses
        quarks = ['u', 'd', 's', 'c', 'b', 't']
        masses = [self.results['masses'][q] for q in quarks]
        colors = ['r', 'b', 'g', 'c', 'm', 'y']

        # Create bar chart with log scale
        ax4.bar(quarks, masses, color=colors)
        ax4.set_yscale('log')

        # Set labels and title
        ax4.set_xlabel('Quark')
        ax4.set_ylabel('Mass (GeV) - Log Scale')
        ax4.set_title('Quark Mass Hierarchy')
        ax4.grid(True, axis='y', linestyle='--', alpha=0.7)

        plt.tight_layout()

        # Save plot
        plot_path = "enhanced_comprehensive_visualization.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

        return plot_path


if __name__ == "__main__":
    # Create model
    model = EnhancedPolynomialModel()

    # Optimize parameters
    results = model.optimize_parameters()

    # Generate report
    report_path = model.generate_report()

    # Generate plots
    running_masses_plot, light_quarks_plot = model.plot_running_masses()
    alpha_s_plot = model.plot_alpha_s()
    polynomial_functions_plot, polynomial_log_plot = model.plot_polynomial_functions()
    geodesic_lengths_plot = model.plot_geodesic_lengths()
    mass_hierarchy_plot = model.plot_mass_hierarchy()
    ckm_matrix_plot = model.plot_ckm_matrix()
    comprehensive_plot = model.create_comprehensive_visualization()

    print(f"Report generated at: {report_path}")
    print(f"Running masses plot: {running_masses_plot}")
    print(f"Light quarks plot: {light_quarks_plot}")
    print(f"Alpha_s plot: {alpha_s_plot}")
    print(f"Polynomial functions plot: {polynomial_functions_plot}")
    print(f"Polynomial functions log plot: {polynomial_log_plot}")
    print(f"Geodesic lengths plot: {geodesic_lengths_plot}")
    print(f"Mass hierarchy plot: {mass_hierarchy_plot}")
    print(f"CKM matrix plot: {ckm_matrix_plot}")
    print(f"Comprehensive visualization: {comprehensive_plot}")

    # Print summary of key results
    print("\nOptimized Parameters:")
    print("Up-type quark coefficients:")
    for i, c in enumerate(model.c_coeffs_up):
        print(f"c_{i} = {c:.6f} GeV")

    print("\nDown-type quark coefficients:")
    for i, c in enumerate(model.c_coeffs_down):
        print(f"c_{i} = {c:.6f} GeV")

    print("\nTop quark enhancement parameters:")
    print(f"Enhancement Factor = {model.top_enhancement_factor:.6f}")
    print(f"Exponent = {model.top_exponent:.6f}")

    print("\nGeneration scaling factors:")
    for i, s in enumerate(model.gen_scale):
        print(f"Generation {i+1}: {s:.6f}")

    print("\nGeodesic Lengths:")
    print(f"L_u = {model.L_u:.6f}")
    print(f"L_d = {model.L_d:.6f}")
    print(f"L_s = {model.L_s:.6f}")
    print(f"L_c = {model.L_c:.6f}")
    print(f"L_b = {model.L_b:.6f}")
    print(f"L_t = {model.L_t:.6f}")

    print("\nCKM Matrix Parameters:")
    print(f"θ₁₂ = {model.theta_12:.6f} rad = {np.degrees(model.theta_12):.4f}°")
    print(f"θ₁₃ = {model.theta_13:.6f} rad = {np.degrees(model.theta_13):.4f}°")
    print(f"θ₂₃ = {model.theta_23:.6f} rad = {np.degrees(model.theta_23):.4f}°")
    print(f"δ_CP = {model.delta_cp:.6f} rad = {np.degrees(model.delta_cp):.4f}°")

    print("\nMass Predictions:")
    for quark in model.quark_info:
        ref_scale = model.quark_info[quark]['ref_scale']
        ref_mass = model.quark_info[quark]['ref_mass']
        pred_mass = results['masses'][quark]
        error = results['errors'][quark]
        print(f"{quark} quark at {ref_scale:.4f} GeV: {pred_mass:.6f} GeV (Reference: {ref_mass:.6f} GeV, Error: {error:.4f}%)")

    print("\nCKM Matrix (Predicted):")
    for i in range(3):
        print(f"[ {model.results['ckm'][i, 0]:.6f} {model.results['ckm'][i, 1]:.6f} {model.results['ckm'][i, 2]:.6f} ]")

    print("\nCKM Matrix (Experimental):")
    for i in range(3):
        print(f"[ {model.ckm_exp[i, 0]:.6f} {model.ckm_exp[i, 1]:.6f} {model.ckm_exp[i, 2]:.6f} ]")

    print("\nCKM Matrix Errors (%):")
    for i in range(3):
        print(f"[ {model.results['ckm_errors'][i, 0]:.4f} {model.results['ckm_errors'][i, 1]:.4f} {model.results['ckm_errors'][i, 2]:.4f} ]")

    print("\nRunning Masses at Different Scales:")
    mu_values = [1.0, 5.0, 91.1876, 173.0]
    running_masses = model.caplculate_running_masses(mu_values)

    print("Scale (GeV) | m_u (GeV) | m_d (GeV) | m_s (GeV) | m_c (GeV) | m_b (GeV) | m_t (GeV)")
    print("-" * 90)
    for i, mu in enumerate(running_masses['mu']):
        print(f"{mu:11.4f} | {running_masses['u'][i]:9.6f} | {running_masses['d'][i]:9.6f} | {running_masses['s'][i]:9.6f} | {running_masses['c'][i]:9.6f} | {running_masses['b'][i]:9.6f} | {running_masses['t'][i]:9.6f}")

    print("\nStrong Coupling Constant at Different Scales:")
    alpha_s_values = model.calculate_alpha_s_values(mu_values)
    for i, mu in enumerate(alpha_s_values['mu']):
        print(f"α_s({mu:.1f} GeV) = {alpha_s_values['alpha_s'][i]:.6f}")


Report generated at: enhanced_model_with_ckm_report.md
Running masses plot: enhanced_running_masses_plot.png
Light quarks plot: enhanced_light_quarks_plot.png
Alpha_s plot: enhanced_alpha_s_plot.png
Polynomial functions plot: enhanced_polynomial_functions_plot.png
Polynomial functions log plot: enhanced_polynomial_functions_log_plot.png
Geodesic lengths plot: enhanced_geodesic_lengths_plot.png
Mass hierarchy plot: enhanced_mass_hierarchy_plot.png
CKM matrix plot: enhanced_ckm_matrix_plot.png
Comprehensive visualization: enhanced_comprehensive_visualization.png

Optimized Parameters:
Up-type quark coefficients:
c_0 = 0.192137 GeV
c_1 = 0.365882 GeV
c_2 = 0.570060 GeV

Down-type quark coefficients:
c_0 = 0.001000 GeV
c_1 = 1.039884 GeV
c_2 = 4.126467 GeV

Top quark enhancement parameters:
Enhancement Factor = 1.000000
Exponent = 1.432468

Generation scaling factors:
Generation 1: 0.009505
Generation 2: 0.639010
Generation 3: 0.100000

Geodesic Lengths:
L_u = 0.084756
L_d = 0.240999
L_s =